Name: Sania Verma
Student ID: 250059406
This repo contains the code snippets for the Malawi document case study

In [ ]:
import sys, pathlib
# find healthcare-thesis/src no matter which folder this notebook is opened from
root = pathlib.Path.cwd()
for _ in range(5):
    if (root / "healthcare-thesis" / "src").exists():
        break
    root = root.parent
SRC = root / "healthcare-thesis" / "src"
sys.path.insert(0, str(SRC))
import warnings; warnings.filterwarnings("ignore")
print("source code folder:", SRC)

## 1 · The document

In [ ]:
from dataset import Dataset
ds = Dataset.load()
doc = ds.documents["Malawi_05"]
print("id       :", doc.doc_id)
print("country  :", doc.country)
print("pages    :", doc.n_pages)
print("characters:", len(doc.text))
print("source   :", doc.source, "(a real PDF - no OCR needed)")

id       : Malawi_05
country  : Malawi
pages    : 128
characters: 194288
source   : native (a real PDF - no OCR needed)

## 2 · What the computer actually sees

Three samples spread through the document. Notice how little of it is about implementation
techniques — this is why "search the document" returns almost nothing useful.

In [ ]:
import re
t = re.sub(r"\s+", " ", doc.text)
for a, b in ((0, 380), (9000, 9380), (18000, 18380)):
    print(f"--- characters {a:,}-{b:,} ---")
    print(t[a:b])
    print()

--- characters 0-380 ---
GOVERNMENT OF THE REPUBLIC OF MALAWI Antimicrobial Resistance Strategy 2017 - 2022 Towards a
world free from fear of untreatable infections LIST OF TABLES.................. ACRONYMS AND
ABBREVIATIONS.....................

--- characters 9,000-9,380 ---
...the AMR Core team; the drafting team comprising of Collins Mitambo, Gift Nangwale and Gerald
Tegha. The MoH wishes to thank Dr. Mirfin Mpundu, Head, Action against Antibiotic Resistance
(ReAct), Africa...

--- characters 18,000-18,380 ---
...including family planning, information and education, and the integration of reproductive
health into national strategies. Target 8: Achieve universal health coverage, including
financial risk protection...

A cover page, an acknowledgements list, and UN goals boilerplate. **None of it mentions an
implementation technique.**

## 3 · From a sentence to a ticked technique

Here are real passages from deeper in the plan, and the technique each one supports.

| What the document says | Technique the expert ticks |
|---|---|
| *"Plan and conduct TV and radio programmes. Distribute awareness and advocacy materials"* | Use mass media |
| *"Objective 3: To improve knowledge of AMR … through in-service training"* | Conduct ongoing training |
| *"Dissemination of surveillance data and research findings. Conferences conducted; number of meetings done"* | Audit and provide feedback |
| *"Develop an all-inclusive plan used for securing and lobbying funding for AMR implementation"* | Access new funding |
| *"…will require the multi-sectoral collaboration … MoH, Ministry of Agriculture, Department of Environmental Affairs and the donor community"* | Build a coalition |

**Not one of these passages contains the technique's name.** The link between the two columns is
a judgement — which is exactly the thing that has to be learned.

## 4 · The row that comes out

In [ ]:
iv = [x for x in ds.interventions("all") if x.doc_id == "Malawi_05"][0]
print("name     :", iv.name)
print("years    : 2017 - 2022")
print("focus    :", iv.focus)
print("level    :", iv.level)
print("maturity :", iv.maturity)
print("techniques:", len(iv.eric_codes))
print()
print("expert's summary:", (iv.summary or "")[:220], "...")

name     : Antimicrobial Resistance strategy (NAP) (2017-22)
years    : 2017 - 2022
focus    : AMR
level    : Macro (International, regional, national)
maturity : Implemented
techniques: 21

expert's summary: Based on the GAP, developed by MoH and drawn from HSSP 2, this plan aims to
make AMR a nationally recognized health issue in Malawi. It focuses on enhancing awareness,
advancing research and surveillance ...

## 5 · How our system did

In [ ]:
from experiments import load
r = load("C4-fusion-dochead")
g = set(r["gold"]["Malawi_05"]); p = set(r["predictions"]["Malawi_05"])
print(f"expert ticked {len(g)}, system ticked {len(p)}")
print(f"  correct       : {len(g & p)}")
print(f"  missed        : {len(g - p)}   {sorted(g - p)}")
print(f"  wrongly added : {len(p - g)}   {sorted(p - g)}")
print(f"  F1 on this document: {200*len(g&p)/(len(g)+len(p)):.1f}")

expert ticked 21, system ticked 21
  correct       : 19
  missed        : 2   ['Conduct educational meetings', 'Develop and organize quality monitoring systems']
  wrongly added : 2   ['Tailor strategies', 'Work with educational institutions']
  F1 on this document: 90.5

> **Be careful with this example.** 90.5 is well above our 54.0 average. It was chosen because
> it shows the pipeline clearly, not because it is typical. Many documents score far lower.
>
> Also — are the two "wrong" ones even wrong? The plan does mention working with a university,
> so *Work with educational institutions* is arguable. This is precisely why a second human
> coder is needed: to measure how much of the remaining error is genuine disagreement between
> trained experts rather than machine error.

## 6 · The tick that proves the whole argument

One of the 19 correct answers was **"Develop academic partnerships"**.

We searched all **194,288 characters** for `academic`, `university`, `college of medicine` —
and found **nothing usable**.

Yet both the human expert and our system ticked it. The evidence is not one retrievable
sentence; it is spread across a hundred pages — research objectives, named institutions, a
drafting team of clinicians and scientists, references to studies — and the judgement is made
from the whole.

**That is "Is Ravi disciplined?" happening in a real document.** It is why the search-based
systems score 9 and 21, and why a method that learns the shape of a document scores 54.

In [ ]:
import re
for kw in ("academic", "university", "college of medicine"):
    hits = len(re.findall(kw, doc.text, re.I))
    print(f'  "{kw}": {hits} usable passages found')
print()
print('yet "Develop academic partnerships" is in the gold standard AND our system got it right.')

  "academic": 0 usable passages found
  "university": 0 usable passages found
  "college of medicine": 0 usable passages found

yet "Develop academic partnerships" is in the gold standard AND our system got it right.